# Ara25 — LED

Public research-preview notebook. Outputs and volatile metadata were removed. Set local dataset/output paths in the configuration cells before running.


In [ ]:
!pip install -U transformers datasets rouge-score nltk

In [ ]:
!pip install -q transformers datasets evaluate rouge-score


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q evaluate

In [ ]:
!pip install -q rouge-score

In [ ]:
!pip install -q sacrebleu

In [ ]:
# ==== LED Long Abstractive Summarization (Arabic) — strong one-cell trainer ====
!pip -q install "transformers>=4.41.0" "evaluate>=0.4.2" "datasets>=2.19.0" "accelerate>=0.34.0" "pandas>=2.0"

import os, re, json, gc, math, random, numpy as np, pandas as pd, torch
from datetime import datetime
from datasets import Dataset
from inspect import signature
import evaluate

from transformers import (
    LEDForConditionalGeneration, LEDTokenizerFast,
    DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments,
    GenerationConfig, set_seed, EarlyStoppingCallback
)

# ---------------- Basics ----------------
def _slug(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "-", s).strip("-._")

MODEL_FAMILY   = "led-long-arabic"
LED_NAME       = "allenai/led-base-16384"      # LED base (long context)
SEED           = 42

JSONL = "/content/drive/MyDrive/PACKED2x2_6144/train_ready_6144.jsonl"
SRC_CANDIDATES = ["Merged_Articles","merged_articles","model_input","text"]
TARGET_KEYS    = ["abstracted","target_summary","summary","target"]

# Budgets
MAX_IN   = 6144
MAX_OUT  = 512
NUM_BEAMS       = 4
NO_REPEAT_NGRAM = 3
LENGTH_PENALTY  = 1.0

# Output dirs (isolated)
RUN_ID   = datetime.now().strftime("%Y%m%d-%H%M%S")
_model_slug = _slug(LED_NAME.split("/")[-1])
_ds_slug    = _slug(os.path.splitext(os.path.basename(JSONL))[0])
RUN_NAME = f"{MODEL_FAMILY}-{_model_slug}-{_ds_slug}-{RUN_ID}"
OUT_ROOT = "/content/drive/MyDrive/sum_runs_led"
OUT      = f"{OUT_ROOT}/{RUN_NAME}"
for sub in ["checkpoints","logs","preds","metrics","artifacts"]:
    os.makedirs(f"{OUT}/{sub}", exist_ok=True)

print("RUN_NAME:", RUN_NAME)
print("OUT:", OUT)

# ---------------- Env ----------------
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
try: torch.set_float32_matmul_precision("high")
except: pass
device = "cuda" if torch.cuda.is_available() else "cpu"
set_seed(SEED)
if torch.cuda.is_available():
    try: torch.cuda.empty_cache()
    except: pass

# ---------------- Data ----------------
with open(JSONL, "r", encoding="utf-8") as f:
    recs = [json.loads(l) for l in f if l.strip()]
assert recs, f"❌ لا توجد سجلات في {JSONL}"
df = pd.DataFrame(recs)

src_key = next((c for c in SRC_CANDIDATES if c in df.columns), None)
tgt_key = next((k for k in TARGET_KEYS    if k in df.columns), None)
assert src_key is not None, f"❌ لم أجد عمود المصدر من: {SRC_CANDIDATES}"
assert tgt_key is not None, f"❌ لم أجد عمود الملخص الهدف من: {TARGET_KEYS}"

_dup_open = re.compile(r'(<ARTICLE_(\d+)>)\s*(?:\1)+', re.IGNORECASE)
_tag_re   = re.compile(r'</?ARTICLE_\d+>', re.IGNORECASE)
_spc_re   = re.compile(r"\s+")

def clean_src(s: str) -> str:
    s = _dup_open.sub(r'\1', s or "")
    s = _tag_re.sub(" ", s)
    s = _spc_re.sub(" ", s).strip()
    return s

def _tidy(s: str) -> str:
    if not s: return ""
    return _spc_re.sub(" ", s).strip()

ds_raw = Dataset.from_pandas(df, preserve_index=False)

# ---------------- Tokenizer & Model ----------------
tok   = LEDTokenizerFast.from_pretrained(LED_NAME, use_fast=True)
model = LEDForConditionalGeneration.from_pretrained(LED_NAME)

# Align special tokens
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token or "</s>"
model.config.pad_token_id           = tok.pad_token_id
model.config.decoder_start_token_id = tok.bos_token_id or tok.cls_token_id or tok.pad_token_id
model.config.bos_token_id           = tok.bos_token_id or tok.cls_token_id or tok.pad_token_id
model.config.eos_token_id           = tok.eos_token_id or tok.sep_token_id or tok.pad_token_id
model.config.no_repeat_ngram_size   = NO_REPEAT_NGRAM
model.config.num_beams              = NUM_BEAMS
model.config.length_penalty         = LENGTH_PENALTY
model.config.early_stopping         = True
model.config.use_cache              = False

# Memory/perf
try: model.gradient_checkpointing_enable()
except: pass
model = model.to(device)

# ---------------- Helpers: safe decode / id sanitization ----------------
def _sanitize_ids_2d(arr, pad_id: int, vocab_size: int):
    """
    Accepts logits or ids; clips to [0, vocab_size) else pad_id; returns list[list[int]]
    """
    if isinstance(arr, tuple) and len(arr) == 1:
        arr = arr[0]
    if isinstance(arr, np.ndarray) and arr.dtype != object:
        a = arr
        if a.ndim == 3:  # logits
            a = a.argmax(axis=-1)
        a = np.asarray(a, dtype=np.int64)
        a = np.where((a < 0) | (a >= vocab_size), pad_id, a)
        return a.tolist()
    out = []
    for row in arr:
        r = np.asarray(row)
        if r.ndim == 2:  # logits row
            r = r.argmax(axis=-1)
        r = r.astype(np.int64, copy=False)
        r = np.where((r < 0) | (r >= vocab_size), pad_id, r)
        out.append(r.tolist())
    return out

# ---------------- Build DS (snap to 1024 multiple + smarter Global Attention) ----------------
ARTICLE_SPLIT = re.compile(r"</?ARTICLE_\d+>", re.IGNORECASE)

def build_ds(max_in_tokens: int):
    enc_max = int(math.ceil(max_in_tokens / 1024.0) * 1024)

    def tok_batch_local(batch):
        sources = [clean_src(s) for s in batch[src_key]]
        targets = [_tidy(t)       for t in batch[tgt_key]]

        enc = tok(
            sources, max_length=enc_max, truncation=True,
            return_attention_mask=True, return_offsets_mapping=True, padding=False
        )
        dec = tok(text_target=targets, max_length=int(MAX_OUT), truncation=True)

        # Build global attention mask: CLS + token after each ARTICLE tag
        ga = []
        for i, src in enumerate(sources):
            ids = enc["input_ids"][i]
            offs = enc["offset_mapping"][i]
            g = [0]*len(ids)
            if g: g[0] = 1  # CLS global
            for m in ARTICLE_SPLIT.finditer(src or ""):
                start = m.end()
                # find first token starting at/after m.end()
                for j,(a,b) in enumerate(offs):
                    if a >= start:
                        g[j] = 1
                        break
            ga.append(g)

        # Labels with pad=-100
        labels = [[(tid if tid != tok.pad_token_id else -100) for tid in seq] for seq in dec["input_ids"]]

        # Drop offsets to keep features clean
        del enc["offset_mapping"]
        enc["labels"] = labels
        enc["global_attention_mask"] = ga
        return enc

    spl = ds_raw.train_test_split(test_size=0.10, seed=SEED)  # 90/10
    tr_raw, va_raw = spl["train"], spl["test"]
    tr = tr_raw.map(tok_batch_local, batched=True, remove_columns=tr_raw.column_names)
    va = va_raw.map(tok_batch_local, batched=True, remove_columns=va_raw.column_names)
    return tr, va, tr_raw, va_raw

# ---------------- Metrics (robust; no OverflowError) ----------------
rouge = evaluate.load("rouge")
def compute_metrics(eval_pred):
    if hasattr(eval_pred, "predictions"):
        preds = eval_pred.predictions
        labels = eval_pred.label_ids
    else:
        preds, labels = (eval_pred[0], eval_pred[1]) if len(eval_pred) >= 2 else eval_pred

    pred_ids = _sanitize_ids_2d(preds, pad_id=tok.pad_token_id, vocab_size=tok.vocab_size)
    labels = np.asarray(labels)
    labels = np.where(labels == -100, tok.pad_token_id, labels)
    label_ids = _sanitize_ids_2d(labels, pad_id=tok.pad_token_id, vocab_size=tok.vocab_size)

    decoded_preds  = [re.sub(r"\s+", " ", s).strip() for s in tok.batch_decode(pred_ids,  skip_special_tokens=True)]
    decoded_labels = [re.sub(r"\s+", " ", s).strip() for s in tok.batch_decode(label_ids, skip_special_tokens=True)]
    res = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=False)
    return {k: float(res[k]) for k in res.keys()}

# ---------------- TrainingArguments helper (deprecation-safe) ----------------
def make_args(num_epochs: int, out_dir: str, gen_max_len: int,
              label_smoothing=0.1, per_bs=1, grad_acc=16, bf16_ok=False, strategy="epoch"):
    sig_args = signature(Seq2SeqTrainingArguments.__init__).parameters
    eval_key = "eval_strategy" if "eval_strategy" in sig_args else "evaluation_strategy"
    kwargs = dict(
        output_dir=out_dir,
        num_train_epochs=int(num_epochs),
        learning_rate=3e-5,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        optim="adamw_torch",
        weight_decay=0.01,
        max_grad_norm=0.5,
        per_device_train_batch_size=per_bs,
        per_device_eval_batch_size=per_bs,
        gradient_accumulation_steps=grad_acc,
        save_strategy=strategy,
        load_best_model_at_end=True,
        metric_for_best_model="rougeLsum",
        greater_is_better=True,
        logging_steps=20,
        logging_first_step=True,
        report_to=[],
        fp16=(not bf16_ok),
        bf16=bf16_ok,
        group_by_length=True,
        dataloader_num_workers=0,
        save_safetensors=True,
        label_smoothing_factor=label_smoothing,
        generation_num_beams=NUM_BEAMS,
        generation_max_length=int(gen_max_len),
        ddp_find_unused_parameters=False,
        predict_with_generate=True,
    )
    if "dataloader_pin_memory" in sig_args:
        kwargs["dataloader_pin_memory"] = True
    kwargs[eval_key] = strategy

    if "include_for_metrics" in sig_args:
        kwargs["include_for_metrics"] = ["labels"]
    else:
        kwargs["include_inputs_for_metrics"] = False

    if strategy == "steps":
        kwargs.update(dict(save_steps=1000, eval_steps=1000))
    return Seq2SeqTrainingArguments(**kwargs)

# Collator (keep memory sane; 8-multiple padding)
coll = DataCollatorForSeq2Seq(tokenizer=tok, model=model, pad_to_multiple_of=8)

# BF16 if GPU arch >= 8 (Ampere+)
try:
    sm_ver = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
    use_bf16 = torch.cuda.is_available() and sm_ver >= 8
except Exception:
    use_bf16 = False

# ---------------- Length Curriculum: Phase A (2048) ----------------
PHASE_A_MAX_IN = min(2048, MAX_IN)
trA, vaA, tr_rawA, va_rawA = build_ds(PHASE_A_MAX_IN)
args_A = make_args(
    num_epochs=1, out_dir=OUT, gen_max_len=MAX_OUT,
    label_smoothing=0.1, per_bs=1, grad_acc=16, bf16_ok=use_bf16, strategy="epoch"
)
trainer_A = Seq2SeqTrainer(
    model=model, args=args_A,
    train_dataset=trA, eval_dataset=vaA,
    data_collator=coll, compute_metrics=compute_metrics, tokenizer=tok,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)
print(f"🚀 Phase A: MAX_IN={PHASE_A_MAX_IN}")
trainer_A.train()

# ---------------- Length Curriculum: Phase B (6144) ----------------
PHASE_B_MAX_IN = min(6144, MAX_IN)
trB, vaB, tr_rawB, va_rawB = build_ds(PHASE_B_MAX_IN)
args_B = make_args(
    num_epochs=5, out_dir=OUT, gen_max_len=MAX_OUT,
    label_smoothing=0.1, per_bs=1, grad_acc=16, bf16_ok=use_bf16, strategy="epoch"
)
trainer_B = Seq2SeqTrainer(
    model=model, args=args_B,
    train_dataset=trB, eval_dataset=vaB,
    data_collator=coll, compute_metrics=compute_metrics, tokenizer=tok,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)
print(f"🚀 Phase B: MAX_IN={PHASE_B_MAX_IN}")
trainer_B.train()

# ---------------- Save best + GenConfig ----------------
best_dir = f"{OUT}/checkpoints/final_best_{RUN_NAME}"
os.makedirs(best_dir, exist_ok=True)
trainer_B.save_model(best_dir)
tok.save_pretrained(best_dir)
gen_cfg = GenerationConfig(
    num_beams=NUM_BEAMS, do_sample=False, max_new_tokens=int(MAX_OUT), early_stopping=True,
    decoder_start_token_id=model.config.decoder_start_token_id,
    pad_token_id=model.config.pad_token_id,
    bos_token_id=model.config.bos_token_id,
    eos_token_id=model.config.eos_token_id,
    no_repeat_ngram_size=NO_REPEAT_NGRAM, length_penalty=LENGTH_PENALTY,
)
gen_cfg.save_pretrained(best_dir)
print("✅ Saved BEST ->", best_dir)

# ---------------- Quick validation preds ----------------
with torch.inference_mode():
    out = trainer_B.predict(vaB, max_length=None)
pred_ids = out.predictions[0] if isinstance(out.predictions, (tuple, list)) else out.predictions
pred_ids = _sanitize_ids_2d(pred_ids, pad_id=tok.pad_token_id, vocab_size=tok.vocab_size)
decoded_preds = [_tidy(s) for s in tok.batch_decode(pred_ids, skip_special_tokens=True)]

labels_arr = np.where(out.label_ids == -100, tok.pad_token_id, out.label_ids)
labels_arr = _sanitize_ids_2d(labels_arr, pad_id=tok.pad_token_id, vocab_size=tok.vocab_size)
decoded_refs = [_tidy(s) for s in tok.batch_decode(labels_arr, skip_special_tokens=True)]

preds_df = pd.DataFrame({"id": list(range(len(decoded_preds))), "prediction": decoded_preds, "reference": decoded_refs})
preds_path = f"{OUT}/preds/preds_led_{RUN_NAME}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
preds_df.to_csv(preds_path, index=False, encoding="utf-8-sig")
with open(f"{OUT}/metrics/val_metrics_led_{RUN_NAME}.json","w",encoding="utf-8") as f:
    json.dump(out.metrics, f, ensure_ascii=False, indent=2)
print("✅ Saved non-hier predictions ->", preds_path)

# ---------------- Manifest + cleanup ----------------
manifest = {
    "run_name": RUN_NAME, "run_id": RUN_ID, "out_dir": OUT,
    "model_name": LED_NAME, "dataset_file": JSONL, "seed": SEED,
    "max_in_user": int(MAX_IN), "max_out": int(MAX_OUT),
    "phaseA_max_in": int(PHASE_A_MAX_IN), "phaseB_max_in": int(PHASE_B_MAX_IN),
    "num_beams": int(NUM_BEAMS), "no_repeat_ngram": int(NO_REPEAT_NGRAM),
    "length_penalty": float(LENGTH_PENALTY),
}
with open(f"{OUT}/artifacts/run_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

del trA, vaA, trB, vaB, ds_raw, df, recs, preds_df
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("✅ ALL DONE")


In [ ]:
# === LED Hierarchical Generation (Stronger/Longer) — GEN-ONLY | dual-phase decoding ===
!pip -q install "pandas>=2.0" "tqdm" "transformers>=4.41.0"

import os, re, json, math, glob, torch, pandas as pd, numpy as np
from datetime import datetime
from typing import List, Dict, Any, Tuple
from tqdm.auto import tqdm
from transformers import AutoTokenizer, LEDForConditionalGeneration

# ---------- Paths ----------
BASE_DIR = "/content/drive/MyDrive/sum_runs_led"
BEST_DIR_OVERRIDE = ""     # ضع مسار best_dir يدويًا إن أردت (المجلد الذي يحتوي config + tokenizer + model)
TEST_PATH = "/content/drive/MyDrive/test_set.jsonl"

def resolve_best_dir(base_dir: str, best_override: str = "") -> str:
    if best_override and os.path.isdir(best_override):
        return best_override
    try:
        _rn = RUN_NAME  # من خلية التدريب (اختياري)
    except NameError:
        _rn = None
    if _rn:
        cand = f"{base_dir}/{_rn}/checkpoints/final_best_{_rn}"
        if os.path.isdir(cand):
            return cand
    runs = [d for d in glob.glob(os.path.join(base_dir, "led-long-arabic-*")) if os.path.isdir(d)]
    bests = []
    for r in runs:
        bests.extend([p for p in glob.glob(os.path.join(r, "checkpoints", "final_best_*")) if os.path.isdir(p)])
    if not bests:
        raise FileNotFoundError(f"❌ لم أجد أي مجلد final_best_* داخل {base_dir}")
    bests.sort(key=os.path.getmtime, reverse=True)
    return bests[0]

BEST_DIR = resolve_best_dir(BASE_DIR, BEST_DIR_OVERRIDE)
RUN_DIR  = os.path.dirname(os.path.dirname(BEST_DIR))
RUN_NAME = os.path.basename(RUN_DIR)
PRED_DIR = f"{RUN_DIR}/preds"; os.makedirs(PRED_DIR, exist_ok=True)

print("✅ Using BEST_DIR:", BEST_DIR)

# ---------- Limits ----------
MAX_IN        = 6144           # حد إدخال هرمي
MAX_OUT       = 512            # حد الإخراج النهائي (new tokens)
CHUNK_TOKS    = 1024           # LED يعمل جيدًا مع كتل أكبر (مضاعفات 1024)
CHUNK_STRIDE  = 256
MID_OUT       = 200            # إخراج المرحلة الوسطى لكل جزء
MAX_CHUNKS    = 64

# Decoding (أقوى وأطول)
NUM_BEAMS           = 5
NO_REPEAT_NGRAM     = 4
LENGTH_PENALTY      = 1.15
REPETITION_PENALTY  = 1.10
MIN_NEW_TOKENS_FINAL= 200      # إلزام حد أدنى لإطالة الملخص النهائي
MIN_NEW_TOKENS_MID  = 80       # إلزام حد أدنى لملخصات الأجزاء
# Sampling للمرحلة الوسطى لتفادي القصَر/الجفاف
TOP_P               = 0.92
TOP_K               = 50
TEMPERATURE         = 0.9

AR_ONLY = False  # اتركها False إن كان التدريب بدون قيود عربية

# ---------- Env ----------
device = "cuda" if torch.cuda.is_available() else "cpu"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.backends.cuda.matmul.allow_tf32 = True

# ---------- IO ----------
def load_jsonl(path: str):
    recs=[]
    with open(path,"r",encoding="utf-8-sig",errors="replace") as f:
        for ln in f:
            s=ln.strip()
            if not s or s.startswith("```"): continue
            try: recs.append(json.loads(s))
            except: pass
    return recs

def get_source(rec: Dict[str,Any]) -> str:
    return (rec.get("Merged_Articles") or rec.get("merged_articles") or
            rec.get("model_input") or rec.get("text") or "")

items = load_jsonl(TEST_PATH)
assert items, f"❌ لا توجد عناصر في {TEST_PATH}"

# ---------- Clean like training ----------
_dup_open = re.compile(r'(<ARTICLE_(\d+)>)\s*(?:\1)+', re.IGNORECASE)
_tag_re   = re.compile(r'</?ARTICLE_\d+>', re.IGNORECASE)
_spc_re   = re.compile(r"\s+")

def clean_source(s: str) -> str:
    s = _dup_open.sub(r'\1', s or "")
    s = _tag_re.sub(" ", s)
    s = _spc_re.sub(" ", s).strip()
    return s

def clean_output(s: str) -> str:
    s = _spc_re.sub(" ", (s or "")).strip()
    # إزالة تكرار الجُمل المتتالية
    parts = re.split(r"(?<=[\.!\؟\!]|[،؛])\s+", s)
    seen, out = set(), []
    for p in parts:
        q = p.strip()
        if q and q not in seen:
            seen.add(q); out.append(q)
    s = " ".join(out)
    s = re.sub(r"[،,:;]+$", "", s).strip()
    return s

# ---------- Model / Tokenizer ----------
tok = AutoTokenizer.from_pretrained(BEST_DIR, use_fast=True, local_files_only=True)
model = LEDForConditionalGeneration.from_pretrained(BEST_DIR, local_files_only=True)
cfg = model.config

# تأكيد التوكنات الأساسية
if getattr(cfg, "decoder_start_token_id", None) is None and getattr(tok, "bos_token_id", None) is not None:
    cfg.decoder_start_token_id = tok.bos_token_id
if getattr(cfg, "eos_token_id", None) is None and getattr(tok, "eos_token_id", None) is not None:
    cfg.eos_token_id = tok.eos_token_id
if getattr(cfg, "pad_token_id", None) is None and getattr(tok, "pad_token_id", None) is not None:
    cfg.pad_token_id = tok.pad_token_id

# استخدام cache في التوليد
cfg.use_cache = True

# ضبط جلوبال أتِنشن على أول توكن كما هو موصى به لـ LED
def add_global_attention(enc):
    gmask = torch.zeros_like(enc["input_ids"])
    gmask[:, 0] = 1
    enc["global_attention_mask"] = gmask
    return enc

model = model.to(device).eval()

# ---------- Arabic-Only (اختياري) ----------
def build_bad_words_ids_led(tokenizer):
    if not AR_ONLY: return None
    ALLOWED_EXTRA = set(list(".,،؛:؟!()[]{}«»\"'…-–—%+*/=  \t\n\r"))
    ranges = [(0x0600,0x06FF),(0x0750,0x077F),(0x08A0,0x08FF),(0xFB50,0xFDFF),(0xFE70,0xFEFF)]
    def _is_ar(ch):
        cp = ord(ch)
        return any(lo<=cp<=hi for lo,hi in ranges) or ch in ALLOWED_EXTRA
    bad = []
    for ts, tid in tokenizer.get_vocab().items():
        # LED/BART-BPE رموزها ليست WordPiece؛ نفحص المحارف كما هي
        if any((not _is_ar(c)) for c in ts if c.isalpha()):
            bad.append(tid)
    # استثناءات
    excl = {tokenizer.pad_token_id, getattr(cfg, "eos_token_id", -1)}
    return [[i] for i in sorted(set(bad)) if i not in excl]

bad_words_ids = build_bad_words_ids_led(tok)

# ---------- Chunking ----------
MODEL_MAX_CTX = int(getattr(cfg, "max_position_embeddings", 16384))
def _clip_budget(ids: List[int], budget: int) -> List[int]:
    return ids if len(ids) <= budget else ids[:budget]

def _chunk_input_ids(text: str, chunk_size=CHUNK_TOKS, stride=CHUNK_STRIDE, max_chunks=MAX_CHUNKS, user_budget=MAX_IN) -> List[str]:
    ids = tok.encode(text or "", add_special_tokens=True)
    ids = _clip_budget(ids, int(min(user_budget, MODEL_MAX_CTX - 2)))
    chunks = []
    start = 0; n = 0
    while start < len(ids) and n < int(max_chunks):
        end = min(start + int(chunk_size), len(ids))
        sub = ids[start:end]
        # نجبر على end-of-sentence token إن وُجد
        dec = tok.decode(sub, skip_special_tokens=True)
        chunks.append(dec)
        if end == len(ids): break
        start = max(0, end - int(stride))
        n += 1
    return chunks

# ---------- Generation helpers ----------
@torch.inference_mode()
def _generate_mid(texts: List[str], max_new_tokens=MID_OUT) -> List[str]:
    enc = tok(texts, max_length=CHUNK_TOKS, truncation=True, padding=True, return_tensors="pt")
    enc = {k: v.to(device) for k,v in enc.items()}
    enc = add_global_attention(enc)
    kwargs = dict(
        do_sample=True, top_p=float(TOP_P), top_k=int(TOP_K), temperature=float(TEMPERATURE),
        min_new_tokens=int(MIN_NEW_TOKENS_MID),
        max_new_tokens=int(max_new_tokens),
        no_repeat_ngram_size=int(NO_REPEAT_NGRAM),
        repetition_penalty=float(REPETITION_PENALTY),
        eos_token_id=cfg.eos_token_id, pad_token_id=cfg.pad_token_id,
        decoder_start_token_id=cfg.decoder_start_token_id,
    )
    out = model.generate(**enc, **kwargs)
    txt = tok.batch_decode(out, skip_special_tokens=True)
    return [clean_output(t) for t in txt]

@torch.inference_mode()
def _generate_final(texts: List[str], max_new_tokens=MAX_OUT) -> List[str]:
    enc = tok(texts, max_length=MAX_IN, truncation=True, padding=True, return_tensors="pt")
    enc = {k: v.to(device) for k,v in enc.items()}
    enc = add_global_attention(enc)
    kwargs = dict(
        num_beams=int(NUM_BEAMS),
        length_penalty=float(LENGTH_PENALTY),
        min_new_tokens=int(MIN_NEW_TOKENS_FINAL),
        max_new_tokens=int(max_new_tokens),
        no_repeat_ngram_size=int(NO_REPEAT_NGRAM),
        repetition_penalty=float(REPETITION_PENALTY),
        early_stopping=True,
        eos_token_id=cfg.eos_token_id, pad_token_id=cfg.pad_token_id,
        decoder_start_token_id=cfg.decoder_start_token_id,
    )
    if bad_words_ids: kwargs["bad_words_ids"] = bad_words_ids
    out = model.generate(**enc, **kwargs)
    txt = tok.batch_decode(out, skip_special_tokens=True)
    # Fallback إذا كان الإخراج قصيرًا جدًا: أعد التوليد بـ sampling لطول أكبر
    out_fixed = []
    for t, inp in zip(txt, texts):
        if len(t.split()) < MIN_NEW_TOKENS_FINAL // 2:
            t2 = _generate_mid([inp], max_new_tokens=max_new_tokens)[0]
            out_fixed.append(t2 if len(t2.split()) > len(t.split()) else t)
        else:
            out_fixed.append(t)
    return [clean_output(t) for t in out_fixed]

@torch.inference_mode()
def hierarchical_summarize(text: str, mid_out=MID_OUT, final_out=MAX_OUT, user_budget=MAX_IN) -> Tuple[str, list]:
    base = clean_source(text)
    chunks = _chunk_input_ids(base, CHUNK_TOKS, CHUNK_STRIDE, MAX_CHUNKS, user_budget=user_budget)
    if not chunks:
        return _generate_final([base], max_new_tokens=int(final_out))[0], []
    inter = _generate_mid(chunks, max_new_tokens=int(mid_out))
    bridge = " [SEP] ".join(inter)  # فاصل بسيط بين ملخصات الأجزاء
    final  = _generate_final([bridge], max_new_tokens=int(final_out))[0]
    return final, inter

# ---------- Run on test set ----------
rows = []
for i, rec in tqdm(list(enumerate(items)), total=len(items), desc="Generating (LED Hier v2 Long)"):
    src = get_source(rec)
    pred, inter_list = hierarchical_summarize(src, MID_OUT, MAX_OUT, user_budget=MAX_IN)
    rows.append({
        "id": rec.get("id", i),
        "title": rec.get("title", ""),
        "prediction": pred,
        "n_inter_chunks": len(inter_list),
        "intermediate_bridge_len": sum(len(x.split()) for x in inter_list),
    })

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
pred_path = f"{PRED_DIR}/preds_led_clean_v2_{RUN_NAME}_{ts}.csv"
pd.DataFrame(rows).to_csv(pred_path, index=False, encoding="utf-8-sig")

print("✅ Saved predictions ->", pred_path)
print(f"📦 Count: {len(rows)} | Run: {RUN_NAME} | Beams={NUM_BEAMS} | mid={MID_OUT} | final={MAX_OUT} | min_final={MIN_NEW_TOKENS_FINAL} | AR_ONLY={AR_ONLY}")


In [ ]:
# =================== Comprehensive Evaluation for LED Hier Generation ===================
!pip -q install "pandas>=2.0" "evaluate==0.4.2" "rouge_score==0.1.2" "sacrebleu==2.4.0" \
                 "bert-score==0.3.13" "transformers>=4.41.0" "pot>=0.9.3" tqdm

import os, re, json, math, glob
from typing import List, Dict, Any, Tuple
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModel
import evaluate
from bert_score import BERTScorer
import ot  # POT

# --------- Locate run & predictions ---------
BASE_DIR = "/content/drive/MyDrive/sum_runs_led"
TEST_PATH = "/content/drive/MyDrive/test_set.jsonl"
PRED_PATH_OVERRIDE = ""  # اختياري: ضع مسار CSV محدد إن أردت

def resolve_run_dir(base_dir: str) -> str:
    try:
        rn = RUN_NAME  # إن وُجد من خلية التدريب
    except NameError:
        rn = None
    if rn:
        rd = f"{base_dir}/{rn}"
        if os.path.isdir(rd): return rd
    # fallback: latest LED run
    runs = [d for d in glob.glob(os.path.join(base_dir, "led-long-arabic-*")) if os.path.isdir(d)]
    assert runs, f"❌ لا توجد تشغيلات داخل {base_dir}"
    runs.sort(key=os.path.getmtime, reverse=True)
    return runs[0]

RUN_DIR = resolve_run_dir(BASE_DIR)
PRED_DIR = f"{RUN_DIR}/preds"
OUT_DIR  = f"{RUN_DIR}/metrics/eval_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(OUT_DIR, exist_ok=True)

def latest_pred_csv(pred_dir: str) -> str:
    cands = []
    # أنماط ملفات LED inference التي استخدمناها
    cands.extend(glob.glob(os.path.join(pred_dir, "preds_led_clean_*.csv")))
    cands.extend(glob.glob(os.path.join(pred_dir, "preds_led_phaseC_*.csv")))
    cands.extend(glob.glob(os.path.join(pred_dir, "preds_hier_*.csv")))
    cands.extend(glob.glob(os.path.join(pred_dir, "preds_led_*.csv")))
    # دعم أي CSV عند الحاجة
    if not cands:
        cands.extend(glob.glob(os.path.join(pred_dir, "*.csv")))
    assert cands, f"❌ لم أجد ملفات CSV داخل {pred_dir}"
    cands.sort(key=os.path.getmtime, reverse=True)
    return cands[0]

PRED_PATH = PRED_PATH_OVERRIDE if PRED_PATH_OVERRIDE else latest_pred_csv(PRED_DIR)

print("✅ RUN_DIR :", RUN_DIR)
print("📝 PRED   :", PRED_PATH)
print("🧪 TEST   :", TEST_PATH)
print("📊 OUT    :", OUT_DIR)

# --------- Env ---------
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = True
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("🚀 Device:", device)

# --------- Helpers ---------
CTRL_PREFIX_RE = re.compile(r"^\s*(?:ل?\s?خص)\s*بالعربية\s+الفصحى\s+بدقة\s+واختصار\s+ووضوح\s*:\s*", re.IGNORECASE)
TAG_RE = re.compile(r"</?ARTICLE_\d+>", re.IGNORECASE)

def sanitize_text(s: str) -> str:
    if not isinstance(s, str): s = "" if s is None else str(s)
    s = TAG_RE.sub(" ", s)
    s = CTRL_PREFIX_RE.sub("", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = s.replace(" ,", ",").replace(" .", ".").replace(" ؛", "؛").replace(" ؟", "؟").replace(" :", ":")
    s = s.replace(" ،","،").replace(" .",".")
    return s

def load_jsonl(path: str) -> List[Dict[str, Any]]:
    recs = []
    with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
        for raw in f:
            s = raw.strip()
            if not s or s.startswith("```"): continue
            try:
                obj = json.loads(s)
                if isinstance(obj, dict): recs.append(obj)
            except json.JSONDecodeError:
                pass
    return recs

# --------- Load test + refs ---------
items = load_jsonl(TEST_PATH)
assert items, f"❌ لا يوجد عناصر في {TEST_PATH}"
print("🧪 Loaded items:", len(items))

ref1 = [sanitize_text(x.get("target_summary", "")) for x in items]
ref2 = [sanitize_text(x.get("target_summary_2", "")) for x in items]
has_r2 = [bool(r.strip()) for r in ref2]
num_r2 = sum(1 for x in has_r2 if x)
print(f"Ref1 count: {len(ref1)} | Ref2 non-empty: {num_r2}/{len(ref2)}")

# --------- Load predictions CSV ---------
dfp = pd.read_csv(PRED_PATH)
pred_col = None
for c in ["prediction","pred","output","generated","summary","generated_summary"]:
    if c in dfp.columns: pred_col = c; break
assert pred_col is not None, f"❌ لم أجد عمود التنبؤات في CSV. الأعمدة: {list(dfp.columns)}"

# mapping by id if exists
id2pred = {}
if "id" in dfp.columns:
    for _, row in dfp.iterrows():
        id2pred[str(row["id"])] = sanitize_text(str(row[pred_col]))
    preds = []
    for i, rec in enumerate(items):
        key = str(rec.get("id", i))
        preds.append(id2pred.get(key, ""))
else:
    preds = [sanitize_text(x) for x in dfp[pred_col].astype(str).tolist()]
    if len(preds) < len(items):
        preds += [""] * (len(items) - len(preds))
preds = preds[:len(items)]

print("✅ Predictions loaded:", len(preds))
nonempty = sum(1 for p in preds if p.strip())
avg_words = float(np.mean([len(p.split()) for p in preds])) if preds else 0.0
print(f"🔎 Pred diagnostics -> non-empty: {nonempty}/{len(preds)} | avg_words: {avg_words:.1f}")

# --------- Build refs structures ---------
refs_multi = []
for i in range(len(items)):
    rlist = []
    if ref1[i].strip(): rlist.append(ref1[i])
    if ref2[i].strip(): rlist.append(ref2[i])
    if not rlist: rlist = [""]  # ضمان وجود مرجع
    refs_multi.append(rlist)

# =================== ROUGE ===================
rouge = evaluate.load("rouge")
def rouge_scores(pred, ref):
    sc = rouge.compute(predictions=pred, references=ref, use_stemmer=False)
    return {k: float(sc[k]) for k in ["rouge1","rouge2","rougeL","rougeLsum"]}

rouge_ref1 = rouge_scores(preds, ref1)
rouge_ref2 = rouge_scores(preds, [r if r else "" for r in ref2]) if num_r2>0 else {k: None for k in ["rouge1","rouge2","rougeL","rougeLsum"]}

# best-over-refs (ROUGE-Lsum per-sample)
rouge1_list = []; rouge2_list = []; rougeL_list = []; rougeLsum_list = []; best_ref_ix=[]
for i in tqdm(range(len(items)), desc="ROUGE best-over-refs"):
    rs = [x for x in [ref1[i], ref2[i]] if x] or [""]
    best = {"r1":0.0,"r2":0.0,"rL":0.0,"rLsum":0.0}; bix=0
    for j, r in enumerate(rs):
        sc = rouge.compute(predictions=[preds[i]], references=[r], use_stemmer=False)
        if sc["rougeLsum"] > best["rLsum"]:
            best = {"r1":float(sc["rouge1"]), "r2":float(sc["rouge2"]), "rL":float(sc["rougeL"]), "rLsum":float(sc["rougeLsum"])}
            bix=j
    rouge1_list.append(best["r1"]); rouge2_list.append(best["r2"])
    rougeL_list.append(best["rL"]); rougeLsum_list.append(best["rLsum"])
    best_ref_ix.append(bix)
rouge_best = {
    "rouge1": float(np.mean(rouge1_list)),
    "rouge2": float(np.mean(rouge2_list)),
    "rougeL": float(np.mean(rougeL_list)),
    "rougeLsum": float(np.mean(rougeLsum_list)),
}
rouge_avg = {k: None if rouge_ref2[k] is None else (rouge_ref1[k] + rouge_ref2[k]) / 2.0 for k in rouge_ref1}

# =================== SacreBLEU / chrF ===================
sacrebleu = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def wrap_single(refs):  # [[ref_i] ...]
    return [[r] for r in refs]

bleu_multi = sacrebleu.compute(predictions=preds, references=refs_multi)["score"]
bleu_r1    = sacrebleu.compute(predictions=preds, references=wrap_single(ref1))["score"]
bleu_r2    = sacrebleu.compute(predictions=preds, references=wrap_single([r if r else "" for r in ref2]))["score"] if num_r2>0 else None
bleu_avg   = None if bleu_r2 is None else (bleu_r1 + bleu_r2) / 2.0

chrf_multi = chrf_metric.compute(predictions=preds, references=refs_multi)["score"]
chrf_r1    = chrf_metric.compute(predictions=preds, references=wrap_single(ref1))["score"]
chrf_r2    = chrf_metric.compute(predictions=preds, references=wrap_single([r if r else "" for r in ref2]))["score"] if num_r2>0 else None
chrf_avg   = None if chrf_r2 is None else (chrf_r1 + chrf_r2) / 2.0

# =================== METEOR ===================
try:
    meteor_metric = evaluate.load("meteor")
    met_r1 = meteor_metric.compute(predictions=preds, references=ref1)["meteor"]
    met_r2 = meteor_metric.compute(predictions=preds, references=[r if r else "" for r in ref2])["meteor"] if num_r2>0 else None
    # best-over-refs
    met_best_list = []
    for i in tqdm(range(len(items)), desc="METEOR best-over-refs"):
        rs = [x for x in [ref1[i], ref2[i]] if x] or [""]
        best = 0.0
        for r in rs:
            best = max(best, meteor_metric.compute(predictions=[preds[i]], references=[r])["meteor"])
        met_best_list.append(best)
    met_best = float(np.mean(met_best_list))
except Exception as e:
    print("METEOR skipped:", e)
    met_r1 = met_r2 = met_best = None

# =================== BERTScore (per-ref + best) ===================
print("\n🎯 BERTScore (per-ref + best)")
BERT_MODEL = "xlm-roberta-large"
scorer = BERTScorer(model_type=BERT_MODEL, lang="ar", rescale_with_baseline=False, idf=True, device=device, batch_size=8)

# احسب IDF من جميع المراجع المتاحة
all_refs_flat = [r for rs in refs_multi for r in rs]
scorer.compute_idf(all_refs_flat)

def bert_over(preds, refs):
    P,R,F = scorer.score(preds, refs)
    return float(torch.mean(P).item()), float(torch.mean(R).item()), float(torch.mean(F).item())

bP1,bR1,bF1 = bert_over(preds, ref1)
if num_r2>0:
    bP2,bR2,bF2 = bert_over(preds, [r if r else "" for r in ref2])
else:
    bP2=bR2=bF2=None

P1,R1,F1 = scorer.score(preds, ref1)
if num_r2>0:
    P2,R2,F2 = scorer.score(preds, [r if r else "" for r in ref2])
bP_best=[]; bR_best=[]; bF_best=[]
for i in range(len(items)):
    if num_r2>0 and F2[i] > F1[i]:
        bP_best.append(float(P2[i])); bR_best.append(float(R2[i])); bF_best.append(float(F2[i]))
    else:
        bP_best.append(float(P1[i])); bR_best.append(float(R1[i])); bF_best.append(float(F1[i]))
bP_best = float(np.mean(bP_best)); bR_best = float(np.mean(bR_best)); bF_best = float(np.mean(bF_best))

# =================== MoverScore (OT) ===================
print("\n⚖️  MoverScore (per-ref + best)")
EMB_MODEL = "xlm-roberta-large"
MAX_LEN   = 512
tok_emb = AutoTokenizer.from_pretrained(EMB_MODEL)
mdl_emb = AutoModel.from_pretrained(EMB_MODEL).to(device).eval()

@torch.no_grad()
def embed_tokens(text: str, max_len: int = MAX_LEN) -> Tuple[torch.Tensor, List[int]]:
    out = tok_emb(text or "", return_tensors="pt", truncation=True, max_length=max_len, add_special_tokens=True)
    out = {k: v.to(device) for k, v in out.items()}
    hs = mdl_emb(**out).last_hidden_state.squeeze(0)
    ids = out["input_ids"].squeeze(0).tolist()
    special = set(t for t in [getattr(tok_emb, "cls_token_id", None),
                              getattr(tok_emb, "sep_token_id", None),
                              getattr(tok_emb, "pad_token_id", None)] if t is not None)
    keep = [i for i, tid in enumerate(ids) if tid not in special] or list(range(len(ids)))
    return hs[keep], [ids[i] for i in keep]

def build_idf_from_texts(texts: List[str], max_len: int = MAX_LEN) -> dict:
    from collections import Counter, defaultdict
    df_counter = Counter()
    for t in texts:
        ids = tok_emb(t or "", truncation=True, max_length=max_len, add_special_tokens=True)["input_ids"]
        df_counter.update(set(ids))
    N = max(1, len(texts))
    idf = defaultdict(lambda: 0.0)
    for tid, dfv in df_counter.items():
        idf[tid] = math.log((N + 1) / (dfv + 1))
    return idf

def cosine_cost(X: torch.Tensor, Y: torch.Tensor) -> torch.Tensor:
    Xn = torch.nn.functional.normalize(X, dim=1)
    Yn = torch.nn.functional.normalize(Y, dim=1)
    sim = Xn @ Yn.T
    return (1.0 - sim).clamp(min=0.0)

idf_pred = build_idf_from_texts(preds)
idf_ref1 = build_idf_from_texts(ref1)
idf_ref2 = build_idf_from_texts([r if r else "" for r in ref2]) if num_r2>0 else {}

def moverscore_pair(hyp: str, ref: str, idf_h: dict, idf_r: dict) -> float:
    H, ids_h = embed_tokens(hyp, MAX_LEN)
    R, ids_r = embed_tokens(ref, MAX_LEN)
    if H.shape[0] == 0 or R.shape[0] == 0: return 0.0
    a = torch.tensor([idf_h.get(tid, 0.0) for tid in ids_h], dtype=torch.float32)
    b = torch.tensor([idf_r.get(tid, 0.0) for tid in ids_r], dtype=torch.float32)
    if float(a.sum()) == 0: a = torch.ones_like(a)
    if float(b.sum()) == 0: b = torch.ones_like(b)
    a = (a / a.sum()).cpu().numpy()
    b = (b / b.sum()).cpu().numpy()
    C = cosine_cost(H, R).cpu().numpy().astype(np.float64)
    T = ot.emd(a, b, C)
    cost = float((T * C).sum())
    return max(0.0, min(1.0, 1.0 - cost))

ms_r1, ms_r2, ms_best = [], [], []
for i in tqdm(range(len(items)), desc="MoverScore"):
    p = preds[i]
    r1 = ref1[i] if ref1[i] else ""
    s1 = moverscore_pair(p, r1, idf_pred, idf_ref1)
    ms_r1.append(s1)
    if num_r2>0 and ref2[i]:
        s2 = moverscore_pair(p, ref2[i], idf_pred, idf_ref2)
        ms_r2.append(s2)
        ms_best.append(max(s1, s2))
    else:
        ms_best.append(s1)

mover_ref1 = float(np.mean(ms_r1))
mover_ref2 = float(np.mean(ms_r2)) if num_r2>0 and ms_r2 else None
mover_best = float(np.mean(ms_best))

# =================== Summary & Save ===================
summary = {
    "run_dir": RUN_DIR,
    "preds_path": PRED_PATH,
    "refs_path": TEST_PATH,
    "count": len(items),
    "pred_non_empty": nonempty,
    "pred_avg_words": avg_words,
    # ROUGE
    "rouge1_ref1_mean": rouge_ref1["rouge1"],
    "rouge2_ref1_mean": rouge_ref1["rouge2"],
    "rougeL_ref1_mean": rouge_ref1["rougeL"],
    "rougeLsum_ref1_mean": rouge_ref1["rougeLsum"],
    "rouge1_ref2_mean": rouge_ref2["rouge1"] if rouge_ref2["rouge1"] is not None else None,
    "rouge2_ref2_mean": rouge_ref2["rouge2"] if rouge_ref2["rouge2"] is not None else None,
    "rougeL_ref2_mean": rouge_ref2["rougeL"] if rouge_ref2["rougeL"] is not None else None,
    "rougeLsum_ref2_mean": rouge_ref2["rougeLsum"] if rouge_ref2["rougeLsum"] is not None else None,
    "rouge1_best_over_refs": rouge_best["rouge1"],
    "rouge2_best_over_refs": rouge_best["rouge2"],
    "rougeL_best_over_refs": rouge_best["rougeL"],
    "rougeLsum_best_over_refs": rouge_best["rougeLsum"],
    # BLEU / chrF
    "sacrebleu_multi_ref": bleu_multi,
    "sacrebleu_ref1": bleu_r1,
    "sacrebleu_ref2": bleu_r2,
    "sacrebleu_avg": None if bleu_r2 is None else (bleu_r1 + bleu_r2) / 2.0,
    "chrf_multi_ref": chrf_multi,
    "chrf_ref1": chrf_r1,
    "chrf_ref2": chrf_r2,
    "chrf_avg": None if chrf_r2 is None else (chrf_r1 + chrf_r2) / 2.0,
    # METEOR
    "meteor_ref1": met_r1,
    "meteor_ref2": met_r2,
    "meteor_best_over_refs": None if 'met_best' not in locals() else met_best,
    # BERTScore
    "bertscore_ref1": {"P": bP1, "R": bR1, "F1": bF1},
    "bertscore_ref2": ({"P": bP2, "R": bR2, "F1": bF2} if bF2 is not None else None),
    "bertscore_best_over_refs": {"P": bP_best, "R": bR_best, "F1": bF_best},
    # MoverScore
    "moverscore_ref1": mover_ref1,
    "moverscore_ref2": mover_ref2,
    "moverscore_best_over_refs": mover_best,
}

rows = []
for i in range(len(items)):
    row = {
        "id": items[i].get("id", i),
        "title": items[i].get("title", ""),
        "prediction": preds[i],
        "ref1": ref1[i],
        "ref2": ref2[i],
        "rougeLsum_ref1": rouge.compute(predictions=[preds[i]], references=[ref1[i] or ""], use_stemmer=False)["rougeLsum"] if ref1[i] else 0.0,
        "rougeLsum_ref2": rouge.compute(predictions=[preds[i]], references=[ref2[i] or ""], use_stemmer=False)["rougeLsum"] if ref2[i] else 0.0,
        "rougeLsum_best": rougeLsum_list[i],
        "best_ref": "Ref1" if (best_ref_ix[i]==0 or not ref2[i]) else "Ref2",
        "moverscore_best": ms_best[i],
    }
    rows.append(row)

csv_path = os.path.join(OUT_DIR, "eval_report_per_sample.csv")
pd.DataFrame(rows).to_csv(csv_path, index=False, encoding="utf-8-sig")
with open(os.path.join(OUT_DIR, "eval_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\n=== Overall Metrics ===")
for k,v in summary.items():
    if k in ["preds_path","refs_path","run_dir","count"]:
        pass
    print(f"{k}: {v}")

print(f"\n📄 Saved per-sample CSV -> {csv_path}")
print(f"📊 Saved summary JSON   -> {os.path.join(OUT_DIR, 'eval_summary.json')}")
print("✅ Done.")


In [ ]:
# === Top-5 Summaries (Print to Colab) — best by ROUGE-Lsum over Ref1/Ref2 (LED) ===
!pip -q install "rouge_score>=0.1.2" "pandas>=2.0"

import os, re, json, glob, pandas as pd
from datetime import datetime
from rouge_score import rouge_scorer

# -------- مساراتك --------
BASE_DIR = "/content/drive/MyDrive/sum_runs_led"  # تم تعديله لـ LED
TEST_PATH = "/content/drive/MyDrive/test_set.jsonl"
PRED_PATH_OVERRIDE = ""  # ضع مسار CSV يدويًا إن رغبت

# -------- مساعدات --------
CTRL_PREFIX_RE = re.compile(r"^\s*(?:ل?\s?خص)\s*بالعربية\s+الفصحى\s+بدقة\s+واختصار\s+ووضوح\s*:\s*", re.IGNORECASE)
TAG_RE = re.compile(r"</?ARTICLE_\d+>", re.IGNORECASE)
def sanitize(s: str) -> str:
    s = "" if s is None else str(s)
    s = TAG_RE.sub(" ", s)
    s = CTRL_PREFIX_RE.sub("", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = s.replace(" ,", ",").replace(" .", ".").replace(" ؛", "؛").replace(" ؟", "؟").replace(" :", ":")
    s = s.replace(" ،","،").replace(" .",".")
    return s

def load_jsonl(path: str):
    recs=[]
    with open(path,"r",encoding="utf-8-sig",errors="replace") as f:
        for ln in f:
            s=ln.strip()
            if not s or s.startswith("```"): continue
            try: recs.append(json.loads(s))
            except: pass
    return recs

def resolve_run_dir(base_dir: str):
    # استخدم RUN_NAME إن كان متوفّرًا من خلية التدريب
    try:
        rn = RUN_NAME
    except NameError:
        rn = None
    if rn and os.path.isdir(f"{base_dir}/{rn}"):
        return f"{base_dir}/{rn}"
    # أحدث تشغيل لـ LED
    runs = [d for d in glob.glob(os.path.join(base_dir,"led-long-arabic-*")) if os.path.isdir(d)]
    assert runs, f"❌ لا توجد تشغيلات داخل {base_dir}"
    runs.sort(key=os.path.getmtime, reverse=True)
    return runs[0]

def latest_pred_csv(pred_dir: str):
    # أولوية لملفات الجيل الخاصة بـ LED
    cands = []
    cands += glob.glob(os.path.join(pred_dir, "preds_led_clean_*.csv"))
    cands += glob.glob(os.path.join(pred_dir, "preds_led_phaseC_*.csv"))
    cands += glob.glob(os.path.join(pred_dir, "preds_hier_*.csv"))
    cands += glob.glob(os.path.join(pred_dir, "preds_led_*.csv"))
    # أخيرًا أي CSV داخل المجلد
    if not cands:
        cands += glob.glob(os.path.join(pred_dir, "*.csv"))
    assert cands, f"❌ لم أجد ملفات CSV داخل {pred_dir}"
    cands.sort(key=os.path.getmtime, reverse=True)
    return cands[0]

# -------- تحضير البيانات --------
RUN_DIR = resolve_run_dir(BASE_DIR)
PRED_DIR = f"{RUN_DIR}/preds"
PRED_PATH = PRED_PATH_OVERRIDE or latest_pred_csv(PRED_DIR)

items = load_jsonl(TEST_PATH)
assert items, f"❌ لا توجد عناصر في {TEST_PATH}"

dfp = pd.read_csv(PRED_PATH)
pred_col = next((c for c in ["prediction","pred","output","summary","generated","generated_summary"] if c in dfp.columns), None)
assert pred_col, f"❌ لم أجد عمود التنبؤات في CSV. الأعمدة: {list(dfp.columns)}"

# اصطفاف التنبؤات مع عناصر الاختبار
if "id" in dfp.columns:
    id2pred = {str(r["id"]): sanitize(str(r[pred_col])) for _,r in dfp.iterrows()}
    preds = [id2pred.get(str(rec.get("id", i)), "") for i,rec in enumerate(items)]
else:
    preds = [sanitize(x) for x in dfp[pred_col].astype(str).tolist()]
    if len(preds) < len(items): preds += [""]*(len(items)-len(preds))
    preds = preds[:len(items)]

ref1 = [sanitize(x.get("target_summary","")) for x in items]
ref2 = [sanitize(x.get("target_summary_2","")) for x in items]
titles = [sanitize(x.get("title","")) for x in items]

# -------- احسب ROUGE-Lsum واختَر الأفضل --------
scorer = rouge_scorer.RougeScorer(["rougeLsum"], use_stemmer=False)
rows = []
for i,(p,r1,r2) in enumerate(zip(preds, ref1, ref2)):
    s1 = scorer.score(r1 or "", p or "")["rougeLsum"].fmeasure if r1 else 0.0
    s2 = scorer.score(r2 or "", p or "")["rougeLsum"].fmeasure if r2 else 0.0
    best = max(s1, s2); best_ref = "Ref1" if s1 >= s2 else "Ref2"
    rows.append({
        "idx": i, "title": titles[i], "pred": p, "ref1": r1, "ref2": r2,
        "rougeLsum_ref1": s1, "rougeLsum_ref2": s2, "rougeLsum_best": best, "best_ref": best_ref
    })

rows.sort(key=lambda r: r["rougeLsum_best"], reverse=True)
top5 = rows[:5]

# -------- طباعة منسقة على شاشة كولاب --------
print(f"\n====== Top-5 by ROUGE-Lsum (best over refs) ======")
print(f"Run: {os.path.basename(RUN_DIR)} | File: {os.path.basename(PRED_PATH)}\n")
for k, r in enumerate(top5, 1):
    print(f"#{k} | idx={r['idx']} | best_ref={r['best_ref']} | rougeLsum_best={r['rougeLsum_best']:.4f}")
    if r["title"]:
        print("• العنوان:", r["title"])
    print(f"• ROUGE-Lsum Ref1: {r['rougeLsum_ref1']:.4f} | Ref2: {r['rougeLsum_ref2']:.4f}")
    print("— المرجع 1 —")
    print(r["ref1"] if r["ref1"] else "—")
    print("— المرجع 2 —")
    print(r["ref2"] if r["ref2"] else "—")
    print("— المُلخّص المُولَّد —")
    print(r["pred"] if r["pred"] else "—")
    print("-"*100)
